### Ingresos por nivel de instrucción
Limpieza de datos obtenidos de la ENOE por la INEGI, del conjunto de datos: Población ocupada se hizo la consulta: Población ocupada/Periodo encuesta y Sexo/ Nivel de ingresos y Nivel de instrucción (personas conforme a su grado máximo de estudios aprobado).

## 1. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#FAFAFA',
    'axes.facecolor': '#FAFAFA',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

VERDE  = "#49B763"   # Hombre
MORADO  = "#79347F"   # Mujer
NARANJA = "#FA9435"   # Brecha 
ROJO = "#F57474"  # Alertas

print('Librerías cargadas correctamente.')

## 2. Carga y limpieza del CSV

El archivo tiene un encabezado multi-línea (2 filas de cabecera: rango de ingreso + nivel de instrucción)  
y cada celda contiene tres valores separados por `|`: `población|CV|estimación`.

In [ ]:
FILE = 'ingreso_nivel_instruccion.csv'

with open(FILE, 'rb') as f:
    raw = f.read()
text = raw.decode('latin1')
lines = text.split('\n')


ingreso     = [c.strip() for c in lines[4].split(',')]
instruccion = [c.strip() for c in lines[5].split(',')]

# Construir nombres de columna compuestos
col_names = []
for ing, inst in zip(ingreso, instruccion):
    if ing == '' and inst == '':
        col_names.append('_skip_')
    elif inst == '':
        col_names.append(ing)
    elif ing == '':
        col_names.append(inst)
    else:
        col_names.append(f'{ing} | {inst}')

print(f'Columnas detectadas: {len(col_names)}')
print('Primeras 10:', col_names[:10])

In [ ]:
records = []
for line in lines[6:]:
    line = line.strip().lstrip('\r').strip()
    if not line or '|' not in line:
        continue
    parts = line.split(',')
    if len(parts) < 10:
        continue
    periodo = parts[0].strip()
    sexo    = parts[1].strip()
    # Solo filas con sexo válido
    if sexo not in ('Hombre', 'Mujer', 'Total'):
        continue
    row = {'periodo': periodo, 'sexo': sexo}
    for i, col in enumerate(col_names[2:], start=2):
        if i >= len(parts):
            break
        cell = parts[i].strip()
        if '|' in cell:
            sub = cell.split('|')
            try:
                row[col] = float(sub[0].replace(' ', '').replace(',', ''))
            except ValueError:
                row[col] = np.nan
        else:
            try:
                row[col] = float(cell.replace(' ', '').replace(',', '')) if cell else np.nan
            except ValueError:
                row[col] = np.nan
    records.append(row)

df_raw = pd.DataFrame(records)
print(f'Filas cargadas: {len(df_raw)}')
df_raw.head(6)

## 3. Transformación a formato largo (tidy)

Separamos las columnas compuestas en dos dimensiones:  
`nivel_ingreso` y `nivel_instruccion`, con el valor en `poblacion`.

In [ ]:
id_vars = ['periodo', 'sexo']
value_vars = [c for c in df_raw.columns if c not in id_vars]

df_long = df_raw.melt(id_vars=id_vars, value_vars=value_vars,
                       var_name='col_combo', value_name='poblacion')

# Separar la columna compuesta en nivel_ingreso y nivel_instruccion
def split_col(s):
    if ' | ' in s:
        a, b = s.split(' | ', 1)
        return a.strip(), b.strip()
    return s.strip(), 'Total'

df_long[['nivel_ingreso', 'nivel_instruccion']] = pd.DataFrame(
    df_long['col_combo'].apply(split_col).tolist(), index=df_long.index
)
df_long = df_long.drop(columns='col_combo')

# --- Limpiar categorías irrelevantes ---
EXCLUIR_INSTRUCCION = {'No especificado', 'Total', '_skip_'}
EXCLUIR_INGRESO     = {'No especificado', 'No recibe ingresos', 'Total', '_skip_'}

df_long = df_long[
    ~df_long['nivel_instruccion'].isin(EXCLUIR_INSTRUCCION) &
    ~df_long['nivel_ingreso'].isin(EXCLUIR_INGRESO) &
    df_long['sexo'].isin(['Hombre', 'Mujer'])
].copy()

df_long['poblacion'] = pd.to_numeric(df_long['poblacion'], errors='coerce')
df_long = df_long.dropna(subset=['poblacion'])

print(f'Registros en formato largo: {len(df_long):,}')
print('Niveles de instrucción:', df_long['nivel_instruccion'].unique().tolist())
print('Rangos de ingreso:', df_long['nivel_ingreso'].unique().tolist())
df_long.head()

## 4. Diagnóstico de calidad de los datos

In [ ]:
print('=== DIAGNÓSTICO DE DATOS ===')
print(f'Periodos únicos  : {df_long["periodo"].nunique()}')
print(f'Sexos            : {df_long["sexo"].unique()}')
print(f'Niveles instruc. : {df_long["nivel_instruccion"].nunique()}')
print(f'Rangos ingreso   : {df_long["nivel_ingreso"].nunique()}')
print(f'Valores nulos    : {df_long["poblacion"].isna().sum()}')
print(f'Valores <= 0     : {(df_long["poblacion"] <= 0).sum()}')
print()
print('Distribución por sexo (total personas en muestra):')
print(df_long.groupby('sexo')['poblacion'].sum().apply(lambda x: f'{x:,.0f}'))

## 5. Construcción del índice de brecha por nivel de instrucción

Calculamos la **distribución de ingresos** (qué proporción cae en cada rango)  
para hombres y mujeres, separado por nivel de instrucción, usando el promedio de todos los trimestres.

In [ ]:
# Orden de rangos de ingreso (de menor a mayor)
ORDEN_INGRESO = [
    'Hasta un salario mínimo',
    'Más de 1 hasta 2 salarios mínimos',
    'Más de 2 hasta 3 salarios mínimos',
    'Más de 3 hasta 5 salarios mínimos',
    'Más de 5 salarios mínimos',
]
# Peso numérico para calcular un score promedio
PESO_INGRESO = {
    'Hasta un salario mínimo'             : 1,
    'Más de 1 hasta 2 salarios mínimos'   : 1.5,
    'Más de 2 hasta 3 salarios mínimos'   : 2.5,
    'Más de 3 hasta 5 salarios mínimos'   : 4,
    'Más de 5 salarios mínimos'           : 6,
}

ORDEN_INSTRUCCION = [
    'Primaria incompleta',
    'Primaria completa',
    'Secundaria completa',
    'Medio superior y superior',
]

df_filt = df_long[df_long['nivel_ingreso'].isin(ORDEN_INGRESO)].copy()

# Promediar todos los trimestres
df_avg = (
    df_filt
    .groupby(['sexo', 'nivel_instruccion', 'nivel_ingreso'], observed=True)['poblacion']
    .mean()
    .reset_index()
)

# Calcular proporciones dentro de cada (sexo × instrucción)
df_avg['total_grupo'] = df_avg.groupby(['sexo', 'nivel_instruccion'])['poblacion'].transform('sum')
df_avg['prop'] = df_avg['poblacion'] / df_avg['total_grupo']

# Score de ingreso ponderado (proxy del ingreso relativo)
df_avg['peso'] = df_avg['nivel_ingreso'].map(PESO_INGRESO)
df_avg['score_ponderado'] = df_avg['prop'] * df_avg['peso']

score = (
    df_avg.groupby(['sexo', 'nivel_instruccion'])['score_ponderado']
    .sum()
    .reset_index()
    .rename(columns={'score_ponderado': 'score_ingreso'})
)

# Tabla pivote para calcular brecha
score_pivot = score.pivot(index='nivel_instruccion', columns='sexo', values='score_ingreso').reset_index()
score_pivot['brecha_abs']  = score_pivot['Hombre'] - score_pivot['Mujer']
score_pivot['brecha_pct']  = (score_pivot['brecha_abs'] / score_pivot['Hombre']) * 100
score_pivot = score_pivot.set_index('nivel_instruccion').loc[ORDEN_INSTRUCCION].reset_index()

print('Score de ingreso relativo y brecha por nivel de instrucción:')
print(score_pivot.to_string(index=False))

## 6. Análisis: Proporción en ingresos altos (>3 salarios mínimos)

In [ ]:
ALTOS = ['Más de 3 hasta 5 salarios mínimos', 'Más de 5 salarios mínimos']

prop_altos = (
    df_avg
    .assign(ingreso_alto=df_avg['nivel_ingreso'].isin(ALTOS))
    .groupby(['sexo', 'nivel_instruccion', 'ingreso_alto'])['prop']
    .sum()
    .reset_index()
    .query('ingreso_alto == True')
    .drop(columns='ingreso_alto')
    .rename(columns={'prop': 'prop_ingreso_alto'})
)

prop_altos_pivot = prop_altos.pivot(index='nivel_instruccion', columns='sexo', values='prop_ingreso_alto').reset_index()
prop_altos_pivot['brecha_pp'] = (prop_altos_pivot['Hombre'] - prop_altos_pivot['Mujer']) * 100
prop_altos_pivot = prop_altos_pivot.set_index('nivel_instruccion').loc[ORDEN_INSTRUCCION].reset_index()

print('Proporción que gana más de 3 salarios mínimos, por sexo e instrucción:')
for _, row in prop_altos_pivot.iterrows():
    h = row['Hombre'] * 100
    m = row['Mujer']  * 100
    gap = row['brecha_pp']
    print(f"  {row['nivel_instruccion']:35s} → Hombre: {h:5.1f}%  Mujer: {m:5.1f}%  Brecha: {gap:+.1f} pp")

In [ ]:
df_raw.info()

In [ ]:
# Guardamos el CSV limpio listo para el Dashboard
df_long.to_csv('../data/ingreso_nivel_instruccion_clean.csv', index=False)


print(f"CSV guardado: {len(df_long):,} filas × {df_long.shape[1]} columnas")
print(df_long.dtypes)

## 7. Visualizaciones
### 7.1 Distribución de ingresos por sexo y nivel de instrucción

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharey=False)
axes = axes.flatten()

etiquetas_cortas = [
    '≤1 SM', '1–2 SM', '2–3 SM', '3–5 SM', '>5 SM'
]

for idx, instruc in enumerate(ORDEN_INSTRUCCION):
    ax = axes[idx]
    sub = df_avg[df_avg['nivel_instruccion'] == instruc].copy()
    sub['nivel_ingreso'] = pd.Categorical(sub['nivel_ingreso'], categories=ORDEN_INGRESO, ordered=True)
    sub = sub.sort_values('nivel_ingreso')

    x = np.arange(len(ORDEN_INGRESO))
    w = 0.35

    for sx, color, offset in [('Hombre', AZUL, -w/2), ('Mujer', ROSA, w/2)]:
        vals = sub[sub['sexo'] == sx]['prop'].values
        if len(vals) == len(ORDEN_INGRESO):
            ax.bar(x + offset, vals * 100, w, label=sx, color=color, alpha=0.85)

    ax.set_title(instruc, fontsize=11, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(etiquetas_cortas, fontsize=9)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_ylabel('% dentro del grupo', fontsize=9)
    ax.legend(fontsize=8)

fig.suptitle(
    'Distribución de ingresos laborales por sexo y nivel de instrucción\n'
    '(promedio de todos los trimestres disponibles, México – ENOE)',
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig('fig1_distribucion_ingresos.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura 1 guardada.')

### 7.2 Brecha de género: score de ingreso ponderado por nivel educativo

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(score_pivot))
w = 0.32

bars_h = ax.bar(x - w/2, score_pivot['Hombre'], w, color=VERDE, alpha=0.9, label='Hombre')
bars_m = ax.bar(x + w/2, score_pivot['Mujer'],  w, color=MORADO, alpha=0.9, label='Mujer')

# Anotaciones de brecha
for i, row in score_pivot.iterrows():
    ymax = max(row['Hombre'], row['Mujer']) + 0.05
    ax.annotate(
        f"Brecha\n{row['brecha_pct']:.1f}%",
        xy=(i, ymax),
        ha='center', va='bottom', fontsize=8.5,
        color=NARANJA, fontweight='bold'
    )

instruc_labels = [
    'Primaria\nincompleta',
    'Primaria\ncompleta',
    'Secundaria\ncompleta',
    'Medio superior\ny superior',
]
ax.set_xticks(x)
ax.set_xticklabels(instruc_labels, fontsize=10)
ax.set_ylabel('Score de ingreso relativo\n(1 = ≤1 SM, 6 = >5 SM)', fontsize=10)
ax.set_title(
    'Score de ingreso relativo por sexo y nivel de instrucción\n'
    '— La brecha persiste en todos los niveles educativos —',
    fontsize=12, fontweight='bold'
)
ax.legend(fontsize=10)
ax.set_ylim(0, score_pivot[['Hombre','Mujer']].max().max() + 0.5)

plt.tight_layout()
plt.savefig('fig2_score_brecha.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura 2 guardada.')

### 7.3 Proporción que alcanza ingresos altos (>3 SM): hombre vs mujer por nivel educativo

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(prop_altos_pivot))
w = 0.32

ax.bar(x - w/2, prop_altos_pivot['Hombre'] * 100, w, color=VERDE, alpha=0.9, label='Hombre')
ax.bar(x + w/2, prop_altos_pivot['Mujer']  * 100, w, color=MORADO, alpha=0.9, label='Mujer')

for i, row in prop_altos_pivot.iterrows():
    gap = row['brecha_pp']
    ymax = max(row['Hombre'], row['Mujer']) * 100 + 1
    ax.annotate(
        f"+{gap:.1f} pp",
        xy=(i, ymax),
        ha='center', va='bottom', fontsize=9,
        color=NARANJA, fontweight='bold'
    )

ax.set_xticks(x)
ax.set_xticklabels(instruc_labels, fontsize=10)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_ylabel('% con ingresos > 3 salarios mínimos', fontsize=10)
ax.set_title(
    '¿Quién llega a ingresos altos (>3 SM)?\n'
    'Hombres con licenciatura tienen ventaja de varios puntos porcentuales sobre mujeres con el mismo nivel',
    fontsize=11, fontweight='bold'
)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('fig3_prop_ingresos_altos.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura 3 guardada.')

### 7.4 Evolución temporal de la brecha (Medio superior y superior)

In [ ]:
# Calcular score por periodo para nivel educativo más alto
df_sup = df_filt[df_filt['nivel_instruccion'] == 'Medio superior y superior'].copy()
df_sup['peso'] = df_sup['nivel_ingreso'].map(PESO_INGRESO)

totales = df_sup.groupby(['periodo', 'sexo'])['poblacion'].transform('sum')
df_sup['prop'] = df_sup['poblacion'] / totales
df_sup['score_w'] = df_sup['prop'] * df_sup['peso']

score_t = (
    df_sup.groupby(['periodo', 'sexo'])['score_w']
    .sum()
    .reset_index()
    .rename(columns={'score_w': 'score'})
)

# Ordenar periodos cronológicamente
ORDEN_T = score_t['periodo'].unique().tolist()[::-1]  # más antiguo primero
score_t['orden'] = score_t['periodo'].apply(lambda p: ORDEN_T.index(p) if p in ORDEN_T else 999)
score_t = score_t.sort_values('orden')

fig, ax = plt.subplots(figsize=(13, 5))
for sx, color in [('Hombre', VERDE), ('Mujer', MORADO)]:
    sub = score_t[score_t['sexo'] == sx]
    ax.plot(sub['orden'], sub['score'], marker='o', markersize=4,
            label=sx, color=color, linewidth=2)

# Rellenar área de brecha
h_vals = score_t[score_t['sexo'] == 'Hombre'].set_index('orden')['score']
m_vals = score_t[score_t['sexo'] == 'Mujer'].set_index('orden')['score']
idx_com = h_vals.index.intersection(m_vals.index)
ax.fill_between(idx_com, h_vals[idx_com], m_vals[idx_com], alpha=0.12, color=NARANJA, label='Brecha')

# Etiquetas eje X (cada 4 periodos para no saturar)
ticks = list(range(0, len(ORDEN_T), 4))
ax.set_xticks(ticks)
ax.set_xticklabels([ORDEN_T[i] for i in ticks], rotation=35, ha='right', fontsize=8)

ax.set_ylabel('Score de ingreso relativo', fontsize=10)
ax.set_title(
    'Evolución temporal de la brecha salarial en personas con Medio superior y superior\n'
    '— La brecha sigue presente a lo largo del tiempo —',
    fontsize=11, fontweight='bold'
)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('fig4_evolucion_temporal.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura 4 guardada.')

## 8. Tabla resumen final

In [ ]:
tabla = score_pivot.merge(prop_altos_pivot[['nivel_instruccion','brecha_pp']], on='nivel_instruccion')
tabla = tabla.rename(columns={
    'nivel_instruccion': 'Nivel de instrucción',
    'Hombre': 'Score Hombre',
    'Mujer' : 'Score Mujer',
    'brecha_abs': 'Brecha absoluta',
    'brecha_pct': 'Brecha (%)',
    'brecha_pp' : 'Brecha prop. ingresos altos (pp)',
})

styled = (
    tabla[['Nivel de instrucción','Score Hombre','Score Mujer','Brecha (%)','Brecha prop. ingresos altos (pp)']]
    .style
    .format({
        'Score Hombre': '{:.3f}',
        'Score Mujer' : '{:.3f}',
        'Brecha (%)': '{:.1f}%',
        'Brecha prop. ingresos altos (pp)': '{:.1f} pp',
    })
    .background_gradient(subset=['Brecha (%)'], cmap='Reds')
    .set_caption('Brecha de género en ingresos laborales por nivel de instrucción – México (ENOE)')
)
styled